# Advanced Python Properties — Problems with Complete Solutions

This notebook is an advanced practice set on Python **instance properties**. It expands the uploaded lesson's core ideas: the `property` object lives on the class, managed values are usually stored per instance, and dotted access can route through getter, setter, and deleter functions.

You will practice validation, normalization, backing attributes, `getattr` / `setattr` / `delattr`, deletion semantics, instance-dictionary precedence, read-only properties, inheritance, cross-property invariants, canonical storage, auditing, and API design.

## Best-practice rules used throughout

1. Keep the public API simple (`obj.name`) and hide implementation storage behind `_name`-style backing attributes.
2. Use `TypeError` when the *kind* of object is wrong and `ValueError` when a value of an acceptable kind is outside the allowed domain.
3. Reject `bool` explicitly when a true numeric value is required, because `bool` is a subclass of `int`.
4. Centralize validation to avoid duplicated rules.
5. Make computed values read-only unless assignment has a clear, unsurprising meaning.
6. Prefer methods for actions such as `deposit()`, `sell()`, or `send()`; a property setter should usually feel like ordinary state assignment.
7. Test the public interface and edge cases, not just the happy path.
8. Document unusual getter/setter/deleter behavior.

In [1]:
def expect_exception(exc_type, func, *args, **kwargs):
    """Return the exception if the expected exception is raised; otherwise fail."""
    try:
        func(*args, **kwargs)
    except exc_type as ex:
        return ex
    except Exception as ex:
        raise AssertionError(
            f"Expected {exc_type.__name__}, got {type(ex).__name__}: {ex}"
        ) from ex
    raise AssertionError(f"Expected {exc_type.__name__}, but no exception was raised")

## Problem 1 — Refactor a public attribute without breaking callers

A class initially exposes `item.name` as a plain attribute. Existing callers already read and assign `item.name`, so changing the API to `get_name()` / `set_name()` would be disruptive.

Refactor `name` into a property that:
- accepts only strings;
- strips whitespace;
- rejects an empty result;
- keeps the public syntax unchanged.

In [2]:
class CatalogItem:
    def __init__(self, name):
        self.name = name

    @property
    def name(self):
        return self._name

    @name.setter
    def name(self, value):
        if not isinstance(value, str):
            raise TypeError("name must be a string")

        normalized = value.strip()
        if not normalized:
            raise ValueError("name must not be empty")

        self._name = normalized


item = CatalogItem("  Python Book  ")
print(item.name)
item.name = "  Advanced Python  "
print(item.name)

Python Book
Advanced Python


In [3]:
assert item.name == "Advanced Python"
assert item.__dict__ == {"_name": "Advanced Python"}
expect_exception(TypeError, setattr, item, "name", 123)
expect_exception(ValueError, setattr, item, "name", "   ")
assert "name" not in item.__dict__

## Problem 2 — Numeric validation and the `bool` trap

Implement `Progress.percentage` with range `0..100`. Accept `int` and `float`, reject `bool`, store a `float`, and use appropriate exception types.

In [4]:
class Progress:
    def __init__(self, percentage=0):
        self.percentage = percentage

    @property
    def percentage(self):
        return self._percentage

    @percentage.setter
    def percentage(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("percentage must be int or float, but not bool")

        value = float(value)
        if not 0.0 <= value <= 100.0:
            raise ValueError("percentage must be between 0 and 100")

        self._percentage = value


progress = Progress(37)
progress.percentage = 82.5
print(progress.percentage)

82.5


In [5]:
assert progress.percentage == 82.5
assert isinstance(progress.percentage, float)
expect_exception(TypeError, setattr, progress, "percentage", True)
expect_exception(TypeError, setattr, progress, "percentage", "50")
expect_exception(ValueError, setattr, progress, "percentage", -0.1)
expect_exception(ValueError, setattr, progress, "percentage", 100.1)

ValueError('percentage must be between 0 and 100')

## Problem 3 — Read-only property

Create a `User.id` property that can be read but not assigned after construction. Validate that IDs are positive integers and reject `bool`.

Then inspect the property object's `fget`, `fset`, and documentation.

In [6]:
class User:
    def __init__(self, user_id, display_name):
        if isinstance(user_id, bool) or not isinstance(user_id, int):
            raise TypeError("user_id must be an integer")
        if user_id <= 0:
            raise ValueError("user_id must be positive")
        self._id = user_id
        self.display_name = display_name

    @property
    def id(self):
        """Stable numeric identifier."""
        return self._id


user = User(101, "Alex")
print(user.id)
print(User.id)
print("fget:", User.id.fget)
print("fset:", User.id.fset)
print("doc:", User.id.__doc__)

101
fget: <function User.id at 0x000001E49009D620>
fset: None
doc: Stable numeric identifier.


In [7]:
assert user.id == 101
assert isinstance(User.id, property)
assert User.id.fget is not None
assert User.id.fset is None
assert User.id.fdel is None
expect_exception(AttributeError, setattr, user, "id", 999)

AttributeError("property 'id' of 'User' object has no setter")

## Problem 4 — Getter, setter, deleter, and post-deletion behavior

Implement a `Session.token` property. Deleting `session.token` should delete the backing `_token` value, **not** the class-level property definition. Reading after deletion must raise a clear `AttributeError`, and assigning a new token afterward must restore the value.

In [8]:
class Session:
    def __init__(self, token):
        self.token = token

    @property
    def token(self):
        if "_token" not in self.__dict__:
            raise AttributeError("token has been deleted")
        return self._token

    @token.setter
    def token(self, value):
        if not isinstance(value, str):
            raise TypeError("token must be a string")
        value = value.strip()
        if not value:
            raise ValueError("token must not be empty")
        self._token = value

    @token.deleter
    def token(self):
        if "_token" not in self.__dict__:
            raise AttributeError("token is already absent")
        del self._token


session = Session("abc-123")
print(session.token)
del session.token
print(session.__dict__)

abc-123
{}


In [9]:
assert isinstance(Session.token, property)
assert Session.token.fdel is not None
assert "_token" not in session.__dict__
expect_exception(AttributeError, getattr, session, "token")

session.token = "new-token"
assert session.token == "new-token"
delattr(session, "token")
expect_exception(AttributeError, delattr, session, "token")

AttributeError('token is already absent')

## Problem 5 — Prove that `getattr`, `setattr`, and `delattr` trigger accessors

Build a managed property that records each get, set, and delete. Use both dotted syntax and the built-in attribute functions and compare the event log.

In [10]:
class TrackedValue:
    def __init__(self, value):
        self.events = []
        self.value = value

    @property
    def value(self):
        self.events.append(("get", self._value))
        return self._value

    @value.setter
    def value(self, new_value):
        self.events.append(("set", new_value))
        self._value = new_value

    @value.deleter
    def value(self):
        old_value = self._value
        self.events.append(("delete", old_value))
        del self._value


tracked = TrackedValue(10)
_ = tracked.value
tracked.value = 20
_ = getattr(tracked, "value")
setattr(tracked, "value", 30)
delattr(tracked, "value")
print(tracked.events)

[('set', 10), ('get', 10), ('set', 20), ('get', 20), ('set', 30), ('delete', 30)]


In [11]:
assert tracked.events == [
    ("set", 10),
    ("get", 10),
    ("set", 20),
    ("get", 20),
    ("set", 30),
    ("delete", 30),
]
assert "_value" not in tracked.__dict__

## Problem 6 — Instance dictionary collision versus property precedence

Predict the output. A `Person.name` property stores its value in `_name`, but code then manually writes a different key called `"name"` into `person.__dict__`.

Questions:
1. What does `person.name` return?
2. Which dictionary entry changes after `person.name = "Raymond"`?
3. Why does the manually inserted `"name"` key fail to shadow the property?

In [12]:
class Person:
    def __init__(self, name):
        self.name = name

    @property
    def name(self):
        return self._name

    @name.setter
    def name(self, value):
        self._name = value


person = Person("Alex")
person.__dict__["name"] = "John"

print("Before:", person.__dict__)
print("person.name:", person.name)

person.name = "Raymond"

print("After:", person.__dict__)
print("person.name:", person.name)

Before: {'_name': 'Alex', 'name': 'John'}
person.name: Alex
After: {'_name': 'Raymond', 'name': 'John'}
person.name: Raymond


In [13]:
assert person.__dict__ == {"_name": "Raymond", "name": "John"}
assert person.name == "Raymond"
assert isinstance(Person.__dict__["name"], property)

# A property with a setter is a data descriptor.
# Data descriptors have priority over a same-named instance dictionary entry.

## Problem 7 — Computed read-only property

Create a `Rectangle` with validated `width` and `height` properties plus a read-only computed `area`. Changing a dimension must immediately affect `area`; assigning directly to `area` must fail.

In [14]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @staticmethod
    def _validate_dimension(name, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f"{name} must be numeric")
        if value <= 0:
            raise ValueError(f"{name} must be greater than zero")
        return float(value)

    @property
    def width(self):
        return self._width

    @width.setter
    def width(self, value):
        self._width = self._validate_dimension("width", value)

    @property
    def height(self):
        return self._height

    @height.setter
    def height(self, value):
        self._height = self._validate_dimension("height", value)

    @property
    def area(self):
        return self.width * self.height


rect = Rectangle(3, 4)
print(rect.area)
rect.width = 10
print(rect.area)

12.0
40.0


In [15]:
assert rect.area == 40.0
expect_exception(AttributeError, setattr, rect, "area", 999)
expect_exception(TypeError, setattr, rect, "width", True)
expect_exception(ValueError, setattr, rect, "height", 0)

ValueError('height must be greater than zero')

## Problem 8 — Cross-property invariant and safe initialization order

Model `TimeWindow(start, end)` where `start <= end`. Each property is an integer (excluding `bool`). During construction, one backing attribute may not exist yet, so the setters must avoid accidentally reading uninitialized state.

In [16]:
class TimeWindow:
    def __init__(self, start, end):
        self.start = start
        self.end = end

    @staticmethod
    def _validate_int(name, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"{name} must be an integer")
        return value

    @property
    def start(self):
        return self._start

    @start.setter
    def start(self, value):
        value = self._validate_int("start", value)
        if "_end" in self.__dict__ and value > self._end:
            raise ValueError("start cannot be greater than end")
        self._start = value

    @property
    def end(self):
        return self._end

    @end.setter
    def end(self, value):
        value = self._validate_int("end", value)
        if "_start" in self.__dict__ and value < self._start:
            raise ValueError("end cannot be less than start")
        self._end = value


window = TimeWindow(10, 20)
window.end = 25
window.start = 12
print(window.start, window.end)

12 25


In [17]:
assert (window.start, window.end) == (12, 25)

before = window.__dict__.copy()
expect_exception(ValueError, setattr, window, "start", 100)
assert window.__dict__ == before
expect_exception(ValueError, setattr, window, "end", 1)
assert window.__dict__ == before

## Problem 9 — Two properties, one canonical stored representation

Implement `Temperature` with Celsius as the **only stored value**. Expose writable `celsius` and `fahrenheit` properties. Enforce absolute zero in Celsius. Do not store both units because duplicated state can drift out of sync.

In [18]:
class Temperature:
    ABSOLUTE_ZERO_C = -273.15

    def __init__(self, celsius):
        self.celsius = celsius

    @staticmethod
    def _as_number(name, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f"{name} must be numeric")
        return float(value)

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        value = self._as_number("celsius", value)
        if value < self.ABSOLUTE_ZERO_C:
            raise ValueError("temperature cannot be below absolute zero")
        self._celsius = value

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32

    @fahrenheit.setter
    def fahrenheit(self, value):
        value = self._as_number("fahrenheit", value)
        self.celsius = (value - 32) * 5 / 9


temp = Temperature(0)
print(temp.celsius, temp.fahrenheit)
temp.fahrenheit = 212
print(temp.celsius, temp.fahrenheit)

0.0 32.0
100.0 212.0


In [19]:
assert round(temp.celsius, 10) == 100.0
assert round(temp.fahrenheit, 10) == 212.0
assert set(temp.__dict__) == {"_celsius"}
expect_exception(ValueError, setattr, temp, "celsius", -274)
expect_exception(ValueError, setattr, temp, "fahrenheit", -500)

ValueError('temperature cannot be below absolute zero')

## Problem 10 — Normalize canonical text in a setter

Create `EmailContact.email` that strips whitespace and lowercases the address. Use intentionally minimal validation: one `@`, non-empty local/domain parts, and at least one dot in the domain.

This is a property exercise, **not** a claim to implement full RFC email validation.

In [20]:
class EmailContact:
    def __init__(self, email):
        self.email = email

    @property
    def email(self):
        return self._email

    @email.setter
    def email(self, value):
        if not isinstance(value, str):
            raise TypeError("email must be a string")

        normalized = value.strip().lower()
        if normalized.count("@") != 1:
            raise ValueError("email must contain exactly one @")

        local, domain = normalized.split("@")
        if not local:
            raise ValueError("email local part must not be empty")
        if not domain or "." not in domain:
            raise ValueError("email domain must contain a dot")

        self._email = normalized


contact = EmailContact("  Alice.Example@Example.COM ")
print(contact.email)

alice.example@example.com


In [21]:
assert contact.email == "alice.example@example.com"
for bad in ["", "@example.com", "alice@example", "alice@@example.com"]:
    expect_exception(ValueError, setattr, contact, "email", bad)
expect_exception(TypeError, setattr, contact, "email", 123)

TypeError('email must be a string')

## Problem 11 — Override only the getter in a subclass

`Score.score` is writable. A subclass wants the getter to return formatted text while keeping the base setter. If the subclass defines a completely new `@property`, it can accidentally discard the inherited setter.

Use `@Score.score.getter` to replace only the getter and preserve the setter.

In [22]:
class Score:
    def __init__(self, score):
        self.score = score

    @property
    def score(self):
        return self._score

    @score.setter
    def score(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("score must be an integer")
        if not 0 <= value <= 100:
            raise ValueError("score must be between 0 and 100")
        self._score = value


class FormattedScore(Score):
    @Score.score.getter
    def score(self):
        return f"{self._score}/100"


result = FormattedScore(88)
print(result.score)
result.score = 92
print(result.score)

88/100
92/100


In [23]:
assert result.score == "92/100"
assert FormattedScore.score.fset is Score.score.fset
assert FormattedScore.score.fget is not Score.score.fget
expect_exception(ValueError, setattr, result, "score", 101)

ValueError('score must be between 0 and 100')

## Problem 12 — Immutable after first assignment

Implement a `Device.serial_number` property that may be assigned exactly once. The first assignment validates and normalizes a non-empty string; later assignments raise `AttributeError`.

In [24]:
class Device:
    def __init__(self, serial_number):
        self.serial_number = serial_number

    @property
    def serial_number(self):
        if "_serial_number" not in self.__dict__:
            raise AttributeError("serial_number has not been assigned")
        return self._serial_number

    @serial_number.setter
    def serial_number(self, value):
        if "_serial_number" in self.__dict__:
            raise AttributeError("serial_number is immutable once assigned")
        if not isinstance(value, str):
            raise TypeError("serial_number must be a string")

        value = value.strip()
        if not value:
            raise ValueError("serial_number must not be empty")

        self._serial_number = value


device = Device(" SN-0001 ")
print(device.serial_number)

SN-0001


In [25]:
assert device.serial_number == "SN-0001"
expect_exception(AttributeError, setattr, device, "serial_number", "SN-0002")

AttributeError('serial_number is immutable once assigned')

## Problem 13 — Audit successful state changes only

Create `InventoryRecord.quantity` with a change log. Failed assignments must leave both the value and the log untouched. Reassigning the same value should be a no-op.

In [26]:
class InventoryRecord:
    def __init__(self, quantity=0):
        self.audit_log = []
        self._quantity = self._validate_quantity(quantity)

    @staticmethod
    def _validate_quantity(value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("quantity must be an integer")
        if value < 0:
            raise ValueError("quantity cannot be negative")
        return value

    @property
    def quantity(self):
        return self._quantity

    @quantity.setter
    def quantity(self, value):
        value = self._validate_quantity(value)
        old_value = self._quantity
        if value == old_value:
            return
        self._quantity = value
        self.audit_log.append((old_value, value))


inventory = InventoryRecord(5)
inventory.quantity = 8
inventory.quantity = 8
inventory.quantity = 3
print(inventory.quantity)
print(inventory.audit_log)

3
[(5, 8), (8, 3)]


In [27]:
assert inventory.quantity == 3
assert inventory.audit_log == [(5, 8), (8, 3)]
snapshot = (inventory.quantity, list(inventory.audit_log))
expect_exception(ValueError, setattr, inventory, "quantity", -1)
assert inventory.quantity == snapshot[0]
assert inventory.audit_log == snapshot[1]

## Problem 14 — When the correct setter is no setter

A bank withdrawal is an operation, not merely a replacement value. Implement a read-only `balance` property plus `deposit()` and `withdraw()` methods.

The design goal is to show that properties are not a reason to turn every state transition into assignment syntax.

In [28]:
class BankAccount:
    def __init__(self, opening_balance=0):
        self._balance = 0.0
        if opening_balance:
            self.deposit(opening_balance)

    @staticmethod
    def _validate_amount(amount):
        if isinstance(amount, bool) or not isinstance(amount, (int, float)):
            raise TypeError("amount must be numeric")
        amount = float(amount)
        if amount <= 0:
            raise ValueError("amount must be greater than zero")
        return amount

    @property
    def balance(self):
        return self._balance

    def deposit(self, amount):
        amount = self._validate_amount(amount)
        self._balance += amount
        return self._balance

    def withdraw(self, amount):
        amount = self._validate_amount(amount)
        if amount > self._balance:
            raise ValueError("insufficient funds")
        self._balance -= amount
        return self._balance


account = BankAccount(100)
account.deposit(25)
account.withdraw(40)
print(account.balance)

85.0


In [29]:
assert account.balance == 85.0
expect_exception(AttributeError, setattr, account, "balance", 1_000_000)
before = account.balance
expect_exception(ValueError, account.withdraw, before + 1)
assert account.balance == before

## Problem 15 — Introspect and directly invoke accessors

A property object exposes `fget`, `fset`, `fdel`, and `__doc__`. Inspect them, then call the accessor functions directly through the property object.

Direct invocation is educational here; normal application code should generally use dotted access.

In [30]:
class ManagedText:
    def __init__(self, text):
        self.text = text

    @property
    def text(self):
        """Normalized text value."""
        return self._text

    @text.setter
    def text(self, value):
        if not isinstance(value, str):
            raise TypeError("text must be a string")
        self._text = value.strip()

    @text.deleter
    def text(self):
        del self._text


managed = ManagedText(" hello ")
prop = ManagedText.__dict__["text"]

print("Property:", prop)
print("Getter:", prop.fget)
print("Setter:", prop.fset)
print("Deleter:", prop.fdel)
print("Doc:", prop.__doc__)
print("Direct fget:", prop.fget(managed))

prop.fset(managed, " world ")
print("After direct fset:", managed.text)

Property: <property object at 0x000001E4910E4950>
Getter: <function ManagedText.text at 0x000001E491100720>
Setter: <function ManagedText.text at 0x000001E4911007C0>
Deleter: <function ManagedText.text at 0x000001E491100860>
Doc: Normalized text value.
Direct fget: hello
After direct fset: world


In [31]:
assert isinstance(prop, property)
assert prop.fget(managed) == "world"
assert prop.__doc__ == "Normalized text value."
prop.fdel(managed)
expect_exception(AttributeError, getattr, managed, "text")

AttributeError("'ManagedText' object has no attribute '_text'")

## Problem 16 — Property factory for repeated validation

Write `positive_number_property(storage_name)` and use it for `Box.length`, `Box.width`, and `Box.height`. Add a read-only computed `volume`.

This technique reduces repetition, but use it judiciously: explicit properties are often easier to read and debug when only a few fields are involved.

In [32]:
def positive_number_property(storage_name):
    public_name = storage_name.removeprefix("_")

    def getter(self):
        return getattr(self, storage_name)

    def setter(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f"{public_name} must be numeric")
        value = float(value)
        if value <= 0:
            raise ValueError(f"{public_name} must be greater than zero")
        setattr(self, storage_name, value)

    return property(
        fget=getter,
        fset=setter,
        doc=f"Positive numeric value stored in {storage_name}.",
    )


class Box:
    length = positive_number_property("_length")
    width = positive_number_property("_width")
    height = positive_number_property("_height")

    def __init__(self, length, width, height):
        self.length = length
        self.width = width
        self.height = height

    @property
    def volume(self):
        return self.length * self.width * self.height


box = Box(2, 3, 4)
print(box.__dict__)
print(box.volume)

{'_length': 2.0, '_width': 3.0, '_height': 4.0}
24.0


In [33]:
assert box.__dict__ == {"_length": 2.0, "_width": 3.0, "_height": 4.0}
assert box.volume == 24.0
expect_exception(ValueError, setattr, box, "length", 0)
expect_exception(TypeError, setattr, box, "width", "3")

TypeError('width must be numeric')

## Problem 17 — Transactional cross-field update

Cross-property invariants become tricky when two values need to change together. Suppose `Range.low <= Range.high`. Updating one property at a time can temporarily violate the invariant even if the final intended pair would be valid.

Implement:
- validated `low` and `high` properties;
- `set_bounds(low, high)` that validates the **pair first**, then commits both values atomically.

In [34]:
class Range:
    def __init__(self, low, high):
        self._low = None
        self._high = None
        self.set_bounds(low, high)

    @staticmethod
    def _validate_number(name, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f"{name} must be numeric")
        return float(value)

    @property
    def low(self):
        return self._low

    @low.setter
    def low(self, value):
        value = self._validate_number("low", value)
        if self._high is not None and value > self._high:
            raise ValueError("low cannot exceed high")
        self._low = value

    @property
    def high(self):
        return self._high

    @high.setter
    def high(self, value):
        value = self._validate_number("high", value)
        if self._low is not None and value < self._low:
            raise ValueError("high cannot be below low")
        self._high = value

    def set_bounds(self, low, high):
        new_low = self._validate_number("low", low)
        new_high = self._validate_number("high", high)

        if new_low > new_high:
            raise ValueError("low cannot exceed high")

        # Commit only after every validation succeeds.
        self._low = new_low
        self._high = new_high


r = Range(0, 10)
r.set_bounds(20, 30)
print(r.low, r.high)

20.0 30.0


In [35]:
assert (r.low, r.high) == (20.0, 30.0)
before = r.__dict__.copy()
expect_exception(ValueError, r.set_bounds, 50, 40)
assert r.__dict__ == before

## Problem 18 — Bug hunt: recursive setter

Why does this setter recurse forever?

```python
@value.setter
def value(self, new_value):
    self.value = new_value
```

**Solution:** assigning to `self.value` from inside the setter invokes the same setter again. Store to a distinct backing attribute such as `self._value`.

In [36]:
class FixedSetter:
    def __init__(self, value):
        self.value = value

    @property
    def value(self):
        return self._value

    @value.setter
    def value(self, new_value):
        self._value = new_value


fixed = FixedSetter(10)
fixed.value = 20
assert fixed.value == 20

## Problem 19 — Bug hunt: recursive getter

Why does this recurse forever?

```python
@property
def value(self):
    return self.value
```

**Solution:** reading `self.value` inside its own getter invokes the same getter again. Return the backing attribute instead.

In [37]:
class FixedGetter:
    def __init__(self, value):
        self._value = value

    @property
    def value(self):
        return self._value


fg = FixedGetter(123)
assert fg.value == 123

## Problem 20 — Distinguish read-only assignment from invalid writable input

A read-only property naturally raises `AttributeError` on assignment. A writable property should normally use `TypeError` for invalid types and `ValueError` for invalid values.

In [38]:
class Example:
    def __init__(self):
        self._read_only = 10
        self._age = 20

    @property
    def read_only(self):
        return self._read_only

    @property
    def age(self):
        return self._age

    @age.setter
    def age(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("age must be an integer")
        if value < 0:
            raise ValueError("age cannot be negative")
        self._age = value


ex = Example()
expect_exception(AttributeError, setattr, ex, "read_only", 99)
expect_exception(TypeError, setattr, ex, "age", "old")
expect_exception(ValueError, setattr, ex, "age", -1)

ValueError('age cannot be negative')

# Problem 21 — Capstone: production-style inventory item

Build `InventoryItem` with:

### Managed properties
- `sku`: string, stripped, uppercased, non-empty, immutable after first assignment;
- `name`: string, stripped, non-empty;
- `unit_price`: numeric (not `bool`), non-negative, stored as `float`;
- `quantity`: integer (not `bool`), non-negative.

### Computed read-only properties
- `inventory_value = unit_price * quantity`;
- `in_stock = quantity > 0`.

### Domain methods
- `restock(amount)` accepts a positive integer;
- `sell(amount)` accepts a positive integer and prevents overselling.

### Serialization
- `to_dict()` exposes public values without leaking `_sku`, `_name`, `_unit_price`, or `_quantity`.

In [39]:
class InventoryItem:
    def __init__(self, sku, name, unit_price, quantity=0):
        self.sku = sku
        self.name = name
        self.unit_price = unit_price
        self.quantity = quantity

    @staticmethod
    def _non_empty_string(field_name, value, *, upper=False):
        if not isinstance(value, str):
            raise TypeError(f"{field_name} must be a string")
        value = value.strip()
        if not value:
            raise ValueError(f"{field_name} must not be empty")
        return value.upper() if upper else value

    @staticmethod
    def _non_negative_number(field_name, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError(f"{field_name} must be numeric")
        value = float(value)
        if value < 0:
            raise ValueError(f"{field_name} cannot be negative")
        return value

    @staticmethod
    def _non_negative_int(field_name, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"{field_name} must be an integer")
        if value < 0:
            raise ValueError(f"{field_name} cannot be negative")
        return value

    @staticmethod
    def _positive_int(field_name, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"{field_name} must be an integer")
        if value <= 0:
            raise ValueError(f"{field_name} must be positive")
        return value

    @property
    def sku(self):
        return self._sku

    @sku.setter
    def sku(self, value):
        if "_sku" in self.__dict__:
            raise AttributeError("sku is immutable once assigned")
        self._sku = self._non_empty_string("sku", value, upper=True)

    @property
    def name(self):
        return self._name

    @name.setter
    def name(self, value):
        self._name = self._non_empty_string("name", value)

    @property
    def unit_price(self):
        return self._unit_price

    @unit_price.setter
    def unit_price(self, value):
        self._unit_price = self._non_negative_number("unit_price", value)

    @property
    def quantity(self):
        return self._quantity

    @quantity.setter
    def quantity(self, value):
        self._quantity = self._non_negative_int("quantity", value)

    @property
    def inventory_value(self):
        return self.unit_price * self.quantity

    @property
    def in_stock(self):
        return self.quantity > 0

    def restock(self, amount):
        amount = self._positive_int("amount", amount)
        self.quantity = self.quantity + amount
        return self.quantity

    def sell(self, amount):
        amount = self._positive_int("amount", amount)
        if amount > self.quantity:
            raise ValueError(
                f"cannot sell {amount}; only {self.quantity} item(s) in stock"
            )
        self.quantity = self.quantity - amount
        return self.quantity

    def to_dict(self):
        return {
            "sku": self.sku,
            "name": self.name,
            "unit_price": self.unit_price,
            "quantity": self.quantity,
            "inventory_value": self.inventory_value,
            "in_stock": self.in_stock,
        }

    def __repr__(self):
        return (
            f"InventoryItem(sku={self.sku!r}, name={self.name!r}, "
            f"unit_price={self.unit_price!r}, quantity={self.quantity!r})"
        )


product = InventoryItem(
    sku=" py-001 ",
    name=" Advanced Python ",
    unit_price=49.95,
    quantity=10,
)

print(product)
print(product.to_dict())

product.restock(5)
product.sell(3)
print(product.to_dict())

InventoryItem(sku='PY-001', name='Advanced Python', unit_price=49.95, quantity=10)
{'sku': 'PY-001', 'name': 'Advanced Python', 'unit_price': 49.95, 'quantity': 10, 'inventory_value': 499.5, 'in_stock': True}
{'sku': 'PY-001', 'name': 'Advanced Python', 'unit_price': 49.95, 'quantity': 12, 'inventory_value': 599.4000000000001, 'in_stock': True}


In [40]:
assert product.sku == "PY-001"
assert product.name == "Advanced Python"
assert product.quantity == 12
assert product.in_stock is True
assert product.inventory_value == product.unit_price * 12
expect_exception(AttributeError, setattr, product, "sku", "OTHER")
expect_exception(ValueError, product.sell, 13)
expect_exception(ValueError, product.restock, 0)
expect_exception(TypeError, product.restock, 1.5)

public = product.to_dict()
for private_name in ["_sku", "_name", "_unit_price", "_quantity"]:
    assert private_name not in public

# Bonus rapid-fire drills

These are short advanced prompts you can solve by modifying the previous classes.

1. Add a `discount_percent` property constrained to `0..100` and a read-only `discounted_price`.
2. Add an audit trail that stores `(timestamp, old_price, new_price)` only after a successful price change.
3. Add a deleter to an optional `description` property and define what reading it after deletion should mean.
4. Write a subclass that changes only the getter formatting of `unit_price` while preserving validation in the inherited setter.
5. Replace three repetitive properties with a property factory, then decide whether the abstraction actually improves readability.
6. Try inserting the public property name directly into `obj.__dict__` and predict whether it will shadow the property.
7. Write a unit-test matrix for every invalid type and boundary value.
8. Refactor a plain public attribute to a property and verify that existing caller syntax still works unchanged.

# Final review checklist

After completing the notebook, you should be able to explain:

- why the property object is stored on the class while managed values are usually instance-specific;
- how `property(fget, fset, fdel, doc)` relates to `@property`, `@x.setter`, and `@x.deleter`;
- why `_name`-style backing attributes prevent accidental recursive access;
- why `getattr`, `setattr`, and `delattr` still trigger property logic;
- why deleting a property value does not remove the property object from the class;
- why a setter-backed property can take precedence over a same-named instance dictionary entry;
- how to create read-only and immutable-after-first-assignment interfaces;
- how to enforce multi-field invariants safely;
- why canonical storage avoids duplicated state;
- how to override only one accessor in a subclass;
- why some state changes should be methods instead of property setters;
- how to inspect and test `fget`, `fset`, `fdel`, and property documentation.